# 03 — Entraînement du modèle avec LoRA (QLoRA)

**Objectif :** spécialiser `Qwen2.5-Coder-1.5B-Instruct` sur le Text-to-SQL financier, en n'entraînant qu'une toute petite partie de ses paramètres.

**Rappel de la baseline (notebook 02) :** 58,9 % d'execution accuracy. Le modèle maîtrise la syntaxe SQL (92,4 % de requêtes valides), mais un tiers de ses réponses sont des requêtes valides qui renvoient un mauvais résultat.

⚙️ **Avant de commencer :** Exécution → Modifier le type d'exécution → **T4 GPU**.

## Principe : LoRA et QLoRA

**LoRA** : au lieu de modifier une grande matrice de poids **W** (par exemple 1536 × 1536), on gèle W et on apprend une petite correction **ΔW = B·A**, où A et B ont un rang **r** très petit (ici r = 16).

```
W' = W (gelé)  +  B · A (entraîné)
```

**QLoRA** : en plus, le modèle de base gelé est chargé en **4 bits** au lieu de 16. Il occupe environ 4 fois moins de mémoire GPU, ce qui permet l'entraînement sur un T4 gratuit.

| Hyperparamètre | Valeur | Justification |
|---|---|---|
| `r` | 16 | Bon compromis capacité / nombre de paramètres pour une tâche ciblée |
| `lora_alpha` | 32 | Règle courante : 2 × r |
| `target_modules` | toutes les couches linéaires | Meilleurs résultats que l'attention seule (recommandation de l'article QLoRA) |
| `learning_rate` | 2e-4 | Valeur standard pour LoRA (plus élevée qu'un fine-tuning complet) |
| Epochs | 2 | Suffisant sur quelques milliers d'exemples ; on surveille le surapprentissage avec la validation |
| Batch effectif | 16 | 4 exemples × 4 pas d'accumulation de gradient |

## 0. Installation

In [ ]:
import os, sys

REPO_URL = "https://github.com/eya-bouhmida/finance-text2sql-lora.git"
REPO_DIR = "/content/finance-text2sql-lora"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    %cd {REPO_DIR}
    !pip install -q transformers accelerate peft trl bitsandbytes datasets
    sys.path.append(REPO_DIR)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    sys.path.append("..")

DRIVE_DIR = "/content/drive/MyDrive/text2sql-finance-qlora" if IN_COLAB else ".."
DATA_DIR = f"{DRIVE_DIR}/data"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints/qlora-r16"   # checkpoints sur Drive : survivent aux coupures
ADAPTER_DIR = f"{DRIVE_DIR}/adapters/qlora-r16"         # adaptateur final
RESULTS_DIR = f"{REPO_DIR}/results" if IN_COLAB else "../results"

In [ ]:
import json, time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from src import data as D, train as T

print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (active le GPU T4)")

## 1. Chargement des données

In [ ]:
train_records = D.load_jsonl(f"{DATA_DIR}/train.jsonl")
val_records = D.load_jsonl(f"{DATA_DIR}/val.jsonl")

train_ds = T.to_prompt_completion(train_records)
val_ds = T.to_prompt_completion(val_records)
print(f"Train : {len(train_ds)} | Validation : {len(val_ds)}")
train_ds[0]

## 2. Chargement du modèle en 4 bits

In [ ]:
torch.cuda.reset_peak_memory_stats()
model, tokenizer = T.load_quantized_model(T.BASE_MODEL)
print(f"Mémoire GPU après chargement en 4 bits : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

### 2.1 Vérification de la longueur des exemples

Les exemples plus longs que `max_length` (1024 tokens) seraient tronqués, et le modèle apprendrait des requêtes coupées. On vérifie qu'aucun exemple ne dépasse cette limite.

In [ ]:
lengths = [
    len(tokenizer(tokenizer.apply_chat_template(r["messages"], tokenize=False))["input_ids"])
    for r in train_records
]
lengths = pd.Series(lengths)
print(f"Longueur en tokens — médiane : {lengths.median():.0f} | 95e percentile : {lengths.quantile(0.95):.0f} | max : {lengths.max()}")
print(f"Exemples > {T.TRAINING_DEFAULTS['max_length']} tokens : {(lengths > T.TRAINING_DEFAULTS['max_length']).sum()}")

## 3. Configuration LoRA et de l'entraînement

In [ ]:
lora_config = T.make_lora_config()          # valeurs par défaut de src/train.py
training_args = T.make_training_args(CHECKPOINT_DIR)

trainer = T.build_trainer(model, tokenizer, train_ds, val_ds, lora_config, training_args)
trainer.model.print_trainable_parameters()

In [ ]:
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"Paramètres entraînés : {trainable:,} sur {total:,} ({100 * trainable / total:.2f} %)")

## 4. Entraînement

Les checkpoints sont sauvegardés sur Google Drive toutes les 50 étapes. **Si Colab coupe la session**, il suffit de tout réexécuter : l'entraînement reprend automatiquement au dernier checkpoint.

In [ ]:
resume = T.latest_checkpoint(CHECKPOINT_DIR)
print("Reprise depuis :", resume or "début (aucun checkpoint)")

start = time.time()
train_output = trainer.train(resume_from_checkpoint=resume)
train_minutes = (time.time() - start) / 60

print(f"\nDurée : {train_minutes:.1f} min")
print(f"Pic de mémoire GPU : {torch.cuda.max_memory_allocated() / 1e9:.2f} Go")

## 5. Courbes d'apprentissage

- La **loss d'entraînement** doit baisser.
- La **loss de validation** doit baisser aussi, puis se stabiliser. Si elle remonte alors que la loss d'entraînement continue de baisser, c'est du **surapprentissage** : le modèle apprend par cœur. Grâce à `load_best_model_at_end`, on garde automatiquement le checkpoint avec la meilleure loss de validation.

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
train_loss = logs.dropna(subset=["loss"])[["step", "loss"]]
eval_loss = logs.dropna(subset=["eval_loss"])[["step", "eval_loss"]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_loss["step"], train_loss["loss"], label="Entraînement", color="#4C72B0")
ax.plot(eval_loss["step"], eval_loss["eval_loss"], label="Validation", color="#DD8452", marker="o")
ax.set_xlabel("Étape"); ax.set_ylabel("Loss"); ax.set_title("Courbes d'apprentissage")
ax.legend(); plt.tight_layout()
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(f"{RESULTS_DIR}/03_learning_curves.png", dpi=150)
plt.show()

print(f"Meilleure loss de validation : {trainer.state.best_metric:.4f} ({trainer.state.best_model_checkpoint})")

## 6. Sauvegarde de l'adaptateur

In [ ]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_mb = T.dir_size_mb(ADAPTER_DIR)
print(f"Adaptateur sauvegardé dans {ADAPTER_DIR}")
print(f"Taille de l'adaptateur : {adapter_mb} Mo (contre environ 3 000 Mo pour le modèle complet en float16)")

In [ ]:
# Résumé de l'entraînement, versionné dans results/ pour le README et le notebook 04
training_summary = {
    "base_model": T.BASE_MODEL,
    "method": "QLoRA (4-bit NF4)",
    "lora": {k: v for k, v in T.LORA_DEFAULTS.items()},
    "training": {k: T.TRAINING_DEFAULTS[k] for k in
                 ["num_train_epochs", "learning_rate", "per_device_train_batch_size",
                  "gradient_accumulation_steps", "max_length"]},
    "n_train": len(train_ds),
    "n_val": len(val_ds),
    "trainable_params": trainable,
    "total_params": total,
    "trainable_pct": round(100 * trainable / total, 3),
    "train_minutes": round(train_minutes, 1),
    "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "adapter_mb": adapter_mb,
    "best_eval_loss": round(trainer.state.best_metric, 4),
}
with open(f"{RESULTS_DIR}/03_training_summary.json", "w") as f:
    json.dump(training_summary, f, indent=2)
logs.to_csv(f"{RESULTS_DIR}/03_training_logs.csv", index=False)
pd.Series(training_summary)

## 7. Premier essai du modèle fine-tuné

Un rapide test sur quelques questions de **validation**, pour vérifier que le modèle produit bien du SQL. L'évaluation complète sur le jeu de test se fait dans le notebook 04, avec exactement la même procédure que la baseline.

In [ ]:
from src import model as M, evaluate as E

tokenizer.padding_side = "left"     # nécessaire pour la génération
trainer.model.config.use_cache = True
trainer.model.eval()

samples = val_records[:5]
prompts = [r["messages"][:-1] for r in samples]   # sans la réponse
outputs, _ = M.generate(trainer.model, tokenizer, prompts)

for r, out in zip(samples, outputs):
    pred = E.extract_sql(out)
    ok = E.evaluate_one(r["context"], r["gold_sql"], pred)["execution_match"]
    print(f"QUESTION  : {r['question']}")
    print(f"RÉFÉRENCE : {r['gold_sql']}")
    print(f"MODÈLE    : {pred}")
    print(f"Correct   : {'✅' if ok else '❌'}\n" + "-" * 80)

## Conclusion de l'entraînement

*À compléter après exécution, par exemple :*

- Seulement **X %** des paramètres ont été entraînés (environ N millions sur 1,5 milliard).
- L'entraînement a pris **X minutes** sur un GPU T4 gratuit, avec un pic de **X Go** de mémoire.
- L'adaptateur pèse **X Mo**, contre environ 3 Go pour le modèle complet.
- Les courbes montrent … (pas de surapprentissage / léger surapprentissage après l'étape …).

**Prochaine étape :** notebook 04, évaluation sur le jeu de test et comparaison avec la baseline.